In [ ]:
import warnings
warnings.filterwarnings("ignore", message=".*pandas only supports SQLAlchemy.*")
warnings.filterwarnings("ignore", category=RuntimeWarning)

import pymysql
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, clear_output

plt.rcParams['font.sans-serif'] = ['SimHei'] 
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style="white", font='SimHei')

def get_db_connection():
    """建立 StarRocks 数据库连接"""
    return pymysql.connect(
        host='192.168.144.101',
        port=9030,
        user='hadoop',
        password='hadoop',
        charset='utf8'
    )

def fetch_cities():
    """获取 DWD 表中有数据的城市列表"""
    conn = get_db_connection()
    query = """
    SELECT DISTINCT city 
    FROM iceberg_catalog.ershoufang.dwd_ershoufang_fact 
    WHERE city IS NOT NULL AND city != ''
    ORDER BY city
    """
    try:
        df = pd.read_sql(query, conn)
        return df['city'].tolist()
    finally:
        conn.close()

def fetch_orientation_price_data(city_name):
    """
    获取指定城市的朝向均价数据。
    过滤无效数据，并在 SQL 层进行聚合，筛选出该城市数量最多的 TOP 10 朝向。
    """
    conn = get_db_connection()
    query = f"""
    WITH top_orientations AS (
        SELECT orientation
        FROM iceberg_catalog.ershoufang.dwd_ershoufang_fact
        WHERE city = '{city_name}'
          AND orientation IS NOT NULL AND orientation != '' AND orientation != '未知'
        GROUP BY orientation
        ORDER BY COUNT(*) DESC
        LIMIT 10
    )
    SELECT 
        orientation,
        AVG(unit_price) as avg_unit_price
    FROM iceberg_catalog.ershoufang.dwd_ershoufang_fact 
    WHERE city = '{city_name}' 
      AND orientation IN (SELECT orientation FROM top_orientations)
      AND unit_price IS NOT NULL AND unit_price > 0
    GROUP BY orientation
    """
    try:
        df = pd.read_sql(query, conn)
        return df
    finally:
        conn.close()

In [ ]:
def plot_orientation_price_polar(city):
    """绘制极坐标条形图 (朝向均价对比)"""
    df = fetch_orientation_price_data(city)
    
    if df.empty or len(df) < 3:
        plt.figure(figsize=(10, 6))
        plt.text(0.5, 0.5, f"{city} 有效朝向数据不足，无法生成分析图", ha='center', va='center', fontsize=14)
        plt.axis('off')
        plt.show()
        return

    # --- 1. 物理方位逻辑排序 (核心设计) ---
    # 北(N) -> 东北(NE) -> 东(E) -> 东南(SE) -> 南(S) -> 西南(SW) -> 西(W) -> 西北(NW)
    standard_order = ['北', '东北', '东', '东南', '南', '西南', '西', '西北']
    order_map = {opt: i for i, opt in enumerate(standard_order)}
    df['order'] = df['orientation'].map(lambda x: order_map.get(x, 99))
    # 按物理方位排序
    df = df.sort_values(by='order')

    # --- 2. 数据准备与极坐标参数 ---
    labels = df['orientation'].tolist()
    prices = df['avg_unit_price'].values
    num_vars = len(labels)

    # 计算角度位置 (均分 360 度)
    angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False)
    width = 2 * np.pi / num_vars * 0.8

    # --- 3. 颜色渐变映射 ---
    norm = mcolors.Normalize(vmin=prices.min(), vmax=prices.max())
    cmap = cm.get_cmap('RdYlBu_r') 
    colors = cmap(norm(prices))

    # --- 4. 开始绘图 (引入极坐标系) ---
    fig, ax = plt.subplots(figsize=(12, 12), subplot_kw=dict(polar=True))
    ax.set_theta_zero_location("N")  
    ax.set_theta_direction(-1)      

    # 绘制极坐标条形图
    bars = ax.bar(
        angles, prices, 
        width=width, 
        bottom=prices.max() * 0.15, 
        color=colors, 
        edgecolor='white', 
        linewidth=1.2,
        alpha=0.9
    )

    # --- 5. 标注与修饰 ---
    ax.set_xticks(angles)
    ax.set_xticklabels(labels, fontsize=12, fontweight='bold', color='#2C3E50')

    # 计算全市平均单价作为基准线
    city_avg_price = prices.mean()
    # 绘制基准圆环
    ax.plot(
        np.linspace(0, 2 * np.pi, 200), 
        [city_avg_price + prices.max() * 0.15] * 200, 
        color='#7F8C8D', linestyle='--', linewidth=1.5, alpha=0.7, label='热门朝向平均水位'
    )

    # 在柱体末端添加具体的均价数值标签
    y_range = prices.max() - prices.min()
    for angle, price, bar in zip(angles, prices, bars):
        rotation = np.degrees(angle)
        if 90 < rotation < 270:
            rotation += 180 
            
        ax.text(
            angle, 
            price + prices.max() * 0.15 + (y_range * 0.1), 
            f'{int(price):,}', 
            ha='center', va='center', 
            fontsize=11, 
            fontweight='bold',
            color='#333333',
            rotation=rotation,
            rotation_mode='anchor'
        )

    # 图表标题与辅助线设置
    plt.title(f'[{city}] 各朝向房源平均单价溢价分析 (元/㎡)', fontsize=18, pad=60, fontweight='bold')
    ax.set_yticklabels([])
    # 弱化网格线
    ax.grid(color='#E5E7E9', linestyle='--', linewidth=1)
    # 移除极坐标外圈黑色圆环
    ax.spines['polar'].set_visible(False)
    
    ax.legend(loc='lower center', bbox_to_anchor=(0.5, 0.05), frameon=True, fontsize=11)

    plt.tight_layout()
    plt.show()

In [3]:
def render_interactive_dashboard():
    """构建交互式下拉组件"""
    cities = fetch_cities()
    if not cities:
        return
    
    default_city = '北京' if '北京' in cities else cities[0]
    
    city_dropdown = widgets.Dropdown(
        options=cities,
        value=default_city,
        description='📍 分析城市:',
        layout={'width': '250px'}
    )
    
    plot_output = widgets.Output()

    def on_city_change(change):
        if change['type'] == 'change' and change['name'] == 'value':
            with plot_output:
                clear_output(wait=True)
                plot_orientation_price_polar(change['new'])

    city_dropdown.observe(on_city_change)

    display(city_dropdown)
    display(plot_output)
    
    with plot_output:
        plot_orientation_price_polar(default_city)

In [4]:
if __name__ == "__main__":
    render_interactive_dashboard()

Dropdown(description='📍 分析城市:', index=4, layout=Layout(width='250px'), options=('上海', '东莞', '中山', '佛山', '北京', …

Output()